In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(123)

class DummyNeuNet(nn.Module):
    def __init__(self, in_dim: int = 8, out_dim: int = 2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 30)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(30, 15)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(15, out_dim)

    def forward(self, x):
        z1 = self.fc1(x)
        h1 = self.relu1(z1)

        z2 = self.fc2(h1)
        h2 = self.relu2(z2)

        logits = self.fc3(h2)
        return {
            "z1": z1,
            "h1": h1,
            "z2": z2,
            "h2": h2,
            "logits": logits
        }

model = DummyNeuNet()
print(model)

model_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total trainable params: ", model_params)
print(model.fc2.weight)


DummyNeuNet(
  (fc1): Linear(in_features=8, out_features=30, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=30, out_features=15, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=15, out_features=2, bias=True)
)
Total trainable params:  767
Parameter containing:
tensor([[-0.0666,  0.0943,  0.0573, -0.0473, -0.0500, -0.1615, -0.0500, -0.0740,
         -0.0995, -0.1649,  0.1430, -0.1632,  0.1812, -0.0958, -0.0140,  0.1489,
          0.0602, -0.0521, -0.1470, -0.0746,  0.1470, -0.0689,  0.1522, -0.0314,
         -0.0233,  0.0729, -0.0268, -0.0015,  0.1264,  0.0610],
        [-0.0073,  0.0695,  0.1590,  0.0460, -0.0535,  0.0598, -0.0160, -0.1427,
         -0.0705,  0.0830,  0.0060,  0.0674, -0.1069,  0.1726, -0.0762,  0.0389,
         -0.0892, -0.0881,  0.0818, -0.0510, -0.1158, -0.0746,  0.1331,  0.1099,
          0.1111, -0.1558,  0.0860,  0.0456, -0.1228,  0.0058],
        [ 0.0365, -0.0986, -0.0770,  0.1489, -0.0148, -0.0019, -0.1156, -0.1092,
          0.1681,  0.0679

In [3]:
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(123)

X_train = torch.rand(5, 8)
y_train = torch.tensor([0, 0, 1, 1, 0])

X_test = torch.rand(3, 8)
y_test = torch.tensor([1, 1, 0])

class DummyDataset(Dataset):
    def __init__(self, X, y):
        super().__init__()
        self.features = X
        self.labels = y

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
    
    def __len__(self):
        return self.labels.shape[0]
        
train_ds = DummyDataset(X_train, y_train)
test_ds = DummyDataset(X_test, y_test)

train_dataloader = DataLoader(
    train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

test_dataloader = DataLoader(
    test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

print(len(train_dataloader))

for idx, (features, labels) in enumerate(train_dataloader):
    print(f"batch {idx+1}", (features, labels))

3
batch 1 (tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025],
        [0.1186, 0.8274, 0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826]]), tensor([0, 1]))
batch 2 (tensor([[0.2745, 0.6584, 0.2775, 0.8573, 0.8993, 0.0390, 0.9268, 0.7388],
        [0.7179, 0.7058, 0.9156, 0.4340, 0.0772, 0.3565, 0.1479, 0.5331]]), tensor([1, 0]))
batch 3 (tensor([[0.1841, 0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017]]), tensor([0]))


In [4]:
import torch

torch.manual_seed(123)

model = DummyNeuNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
epochs = 3

for ep in range(epochs):
    model.train()

    for idx, (features, labels) in enumerate(train_dataloader):
        output = model(features)
        logits = output["logits"]
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {ep+1:03d}/{epochs:03d}, Batch {idx+1:03d}/{len(train_dataloader):03d}"
            f"train loss: {loss:0.4f}")

    model.eval()


Epoch 001/003, Batch 003/003train loss: 1.0274
Epoch 002/003, Batch 003/003train loss: 0.3583
Epoch 003/003, Batch 003/003train loss: 0.6188


In [5]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)

print(outputs["logits"])
probs = F.softmax(outputs["logits"], dim=1)
print(probs)
predictions = torch.argmax(probs, dim=1)
print(predictions)

def compute_accuracy(model, dataloader):
    model.eval()
    total_samples = 0
    correct = 0

    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            output = model(features)
            logits = output["logits"]
        predictions = torch.argmax(logits, dim=1)
        correct += torch.sum( (predictions == labels) )
        total_samples += len(labels)

    return (correct / total_samples).item()

print(compute_accuracy(model, train_dataloader))
print(compute_accuracy(model, test_dataloader))


tensor([[ 0.7279, -0.1518],
        [ 0.2616,  0.2640],
        [-1.0924,  1.3343],
        [-1.2554,  1.5082],
        [ 0.7871, -0.1919]])
tensor([[0.7068, 0.2932],
        [0.4994, 0.5006],
        [0.0812, 0.9188],
        [0.0593, 0.9407],
        [0.7269, 0.2731]])
tensor([0, 1, 1, 1, 0])
0.800000011920929
1.0


In [6]:
model.eval()
outputs = model(X_train)
print(outputs.keys())

for name, value in outputs.items():
    print(name, value.shape)

print(outputs["h1"][0].shape)
print(outputs["h1"][0])     # activation of sample 0
print(outputs["h1"][:, 3])  # activation of 3rd neurons for all samples
print(outputs["h1"][3, : ])  # activation of all 30 neurons for 3rd sample

print(outputs["z1"][0])
print(outputs["h1"][0]) 

h1 = outputs["h1"]
mean_activation = h1.mean(dim=0)
print("Mean actviation shape:  ", mean_activation.shape)
print(mean_activation)

dict_keys(['z1', 'h1', 'z2', 'h2', 'logits'])
z1 torch.Size([5, 30])
h1 torch.Size([5, 30])
z2 torch.Size([5, 15])
h2 torch.Size([5, 15])
logits torch.Size([5, 2])
torch.Size([30])
tensor([0.6649, 0.0000, 0.1931, 0.0946, 0.7008, 0.0000, 0.2471, 0.0000, 0.1779,
        0.0000, 0.3331, 0.0000, 0.1171, 0.4371, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.4719, 0.0299, 0.2515, 0.0000, 0.0000, 0.2394, 0.2525, 0.1364,
        0.0000, 0.0000, 0.0000], grad_fn=<SelectBackward0>)
tensor([0.0946, 0.5135, 1.1956, 1.4649, 0.0909], grad_fn=<SelectBackward0>)
tensor([0.0000, 0.0000, 1.1556, 1.4649, 0.4665, 0.0000, 0.0000, 0.0000, 0.0000,
        0.7802, 0.1901, 0.0000, 0.2741, 1.2969, 0.0000, 0.7324, 0.0000, 0.0000,
        0.0000, 0.0000, 0.2315, 0.0000, 0.0000, 0.0000, 1.3621, 0.0000, 0.9333,
        0.0000, 0.0000, 0.0000], grad_fn=<SliceBackward0>)
tensor([ 0.6649, -0.2248,  0.1931,  0.0946,  0.7008, -0.1556,  0.2471, -0.0732,
         0.1779, -0.0653,  0.3331, -0.5124,  0.1171,  0.4371, -0

In [7]:
values, indices = torch.sort(       # sort return two lists: values, indicies
    mean_activation,
    descending=True
)
print(values, "\n", indices)

print("Neuron Ranking")
for rank, (neuron_idx, score) in enumerate(zip(indices, values), start=1):
    print(f"Rank {rank:02d} | Neuron {neuron_idx.item():02d} | Mean activation {score.item():0.4f}")

activation_frequency = (
    h1 > 0
).float().mean(dim=0)
print(activation_frequency)

max_activation = h1.max(dim=0).values
print(max_activation)

tensor([0.7645, 0.7483, 0.7477, 0.6719, 0.6375, 0.4826, 0.4021, 0.3864, 0.3191,
        0.2822, 0.2639, 0.2042, 0.1780, 0.1498, 0.1393, 0.1277, 0.0799, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000], grad_fn=<SortBackward0>) 
 tensor([24, 13,  4,  3,  2, 26,  9, 10, 15, 19,  0, 12, 25, 21, 20,  6,  8, 27,
        28, 23, 22, 29, 18, 17, 16, 14, 11,  7,  5,  1])
Neuron Ranking
Rank 01 | Neuron 24 | Mean activation 0.7645
Rank 02 | Neuron 13 | Mean activation 0.7483
Rank 03 | Neuron 04 | Mean activation 0.7477
Rank 04 | Neuron 03 | Mean activation 0.6719
Rank 05 | Neuron 02 | Mean activation 0.6375
Rank 06 | Neuron 26 | Mean activation 0.4826
Rank 07 | Neuron 09 | Mean activation 0.4021
Rank 08 | Neuron 10 | Mean activation 0.3864
Rank 09 | Neuron 15 | Mean activation 0.3191
Rank 10 | Neuron 19 | Mean activation 0.2822
Rank 11 | Neuron 00 | Mean activation 0.2639
Rank 12 | Neuron 12 | Mean activation 0.2042
Rank 13 

In [8]:
for neuron_idx in range(h1.shape[1]):

    print(
        f"Neuron {neuron_idx:02d} | "
        f"mean={mean_activation[neuron_idx]:.4f} | "
        f"freq={activation_frequency[neuron_idx]:.2f} | "
        f"max={max_activation[neuron_idx]:.4f}"
    )

Neuron 00 | mean=0.2639 | freq=0.60 | max=0.6649
Neuron 01 | mean=0.0000 | freq=0.00 | max=0.0000
Neuron 02 | mean=0.6375 | freq=1.00 | max=1.2921
Neuron 03 | mean=0.6719 | freq=1.00 | max=1.4649
Neuron 04 | mean=0.7477 | freq=1.00 | max=1.2332
Neuron 05 | mean=0.0000 | freq=0.00 | max=0.0000
Neuron 06 | mean=0.1277 | freq=0.60 | max=0.3703
Neuron 07 | mean=0.0000 | freq=0.00 | max=0.0000
Neuron 08 | mean=0.0799 | freq=0.40 | max=0.2215
Neuron 09 | mean=0.4021 | freq=0.80 | max=0.7802
Neuron 10 | mean=0.3864 | freq=1.00 | max=0.7431
Neuron 11 | mean=0.0000 | freq=0.00 | max=0.0000
Neuron 12 | mean=0.2042 | freq=1.00 | max=0.3607
Neuron 13 | mean=0.7483 | freq=1.00 | max=1.2969
Neuron 14 | mean=0.0000 | freq=0.00 | max=0.0000
Neuron 15 | mean=0.3191 | freq=0.80 | max=0.7324
Neuron 16 | mean=0.0000 | freq=0.00 | max=0.0000
Neuron 17 | mean=0.0000 | freq=0.00 | max=0.0000
Neuron 18 | mean=0.0000 | freq=0.00 | max=0.0000
Neuron 19 | mean=0.2822 | freq=0.60 | max=0.5443
Neuron 20 | mean=0.1

In [9]:
import torch
torch.manual_seed(123)

N_train = 1000
N_test = 200
input_dim = 8

X_train = torch.rand(N_train, input_dim)
X_test = torch.rand(N_test, input_dim)

y_train = (X_train[:, :4].mean(dim=1) > 0.5).long()
y_test = (X_test[:, :4].mean(dim=1) > 0.5).long()

print(X_train.shape)
print(y_train.shape)

print("Class 0: ", (y_train == 0).sum().item())
print("Class 1: ", (y_train == 1).sum().item())

print("Class 0: ", torch.sum(y_train == 0).item())

torch.Size([1000, 8])
torch.Size([1000])
Class 0:  517
Class 1:  483
Class 0:  517


In [10]:
train_ds = DummyDataset(X_train, y_train)
test_ds = DummyDataset(X_test, y_test)

train_dataloader = DataLoader(
    train_ds,
    batch_size = 32,
    shuffle = True,
    num_workers = 0
)
test_dataloader = DataLoader(
    test_ds,
    batch_size = 32,
    shuffle = True,
    num_workers = 0
)

In [11]:
class DummyNeuNet(nn.Module):
    def __init__(self, in_dim: int = 8, out_dim: int = 2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 100)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(100, 100)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(100, out_dim)

    def forward(self, x):
        z1 = self.fc1(x)
        h1 = self.relu1(z1)

        z2 = self.fc2(h1)
        h2 = self.relu2(z2)

        logits = self.fc3(h2)
        return {
            "z1": z1,
            "h1": h1,
            "z2": z2,
            "h2": h2,
            "logits": logits
        }

model = DummyNeuNet()
print(model)

DummyNeuNet(
  (fc1): Linear(in_features=8, out_features=100, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=100, out_features=100, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=100, out_features=2, bias=True)
)


In [17]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epochs = 20

for ep in range(epochs):
    model.train()

    running_loss = 0.0
    for idx, (features, labels) in enumerate(train_dataloader):
        outputs = model(features)
        logits = outputs["logits"]
        loss = F.cross_entropy(
            logits, labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        # print(f"Epoch {ep+1:03d}/{epochs:03d}, Batch {idx+1:03d}/{len(train_dataloader):03d} train loss: {loss:0.4f}")
        
    avg_loss = (running_loss / len(train_dataloader))
    print(f"Epoch {ep+1:03d}/{epochs:03d} avg train loss: {avg_loss:0.4f}")
        

Epoch 001/020 avg train loss: 0.0366
Epoch 002/020 avg train loss: 0.0368
Epoch 003/020 avg train loss: 0.0484
Epoch 004/020 avg train loss: 0.0224
Epoch 005/020 avg train loss: 0.0571
Epoch 006/020 avg train loss: 0.1360
Epoch 007/020 avg train loss: 0.0328
Epoch 008/020 avg train loss: 0.0414
Epoch 009/020 avg train loss: 0.0576
Epoch 010/020 avg train loss: 0.0231
Epoch 011/020 avg train loss: 0.0579
Epoch 012/020 avg train loss: 0.0174
Epoch 013/020 avg train loss: 0.0450
Epoch 014/020 avg train loss: 0.0224
Epoch 015/020 avg train loss: 0.0142
Epoch 016/020 avg train loss: 0.0166
Epoch 017/020 avg train loss: 0.0097
Epoch 018/020 avg train loss: 0.0120
Epoch 019/020 avg train loss: 0.0142
Epoch 020/020 avg train loss: 0.0753


In [18]:
print("Train accuracy: ", compute_accuracy(model, train_dataloader))
print("Test accuracy: ", compute_accuracy(model, test_dataloader))

Train accuracy:  0.9900000095367432
Test accuracy:  0.9700000286102295


In [27]:
def collect_all_activations(model, dataloader):
    all_h1 = []
    all_h2 = []
    all_labels = []

    with torch.no_grad():
        for (features, labels) in dataloader:
            outputs = model(features)

            all_h1.append( outputs["h1"] )
            all_h2.append( outputs["h2"] )
            all_labels.append( labels )


        h1 = torch.cat(all_h1, dim=0)
        h2 = torch.cat(all_h2, dim=0)
        labels = torch.cat(all_labels, dim=0)

    return h1, h2, labels

h1, h2, labels = collect_all_activations(model, train_dataloader)
print("h1 shape: ", h1.shape, " h2 shape: ", h2.shape, " label shape: ", labels.shape )

h1_class0 = h1[labels==0]
h1_class1 = h1[labels==1]

h2_class0 = h2[labels==0]
h2_class1 = h2[labels==1]

print(h1_class0.shape, " ", h1_class1.shape)
print(h2_class0.shape, " ", h2_class1.shape)


h1 shape:  torch.Size([1000, 100])  h2 shape:  torch.Size([1000, 100])  label shape:  torch.Size([1000])
torch.Size([517, 100])   torch.Size([483, 100])
torch.Size([517, 100])   torch.Size([483, 100])


In [31]:
mean_h1_class0 = h1_class0.mean(dim=0)
mean_h1_class1 = h1_class1.mean(dim=0)
print(mean_h1_class0.shape, " ", mean_h1_class1.shape)
print(mean_h1_class0[2], " ", mean_h1_class1[0])

torch.Size([100])   torch.Size([100])
tensor(0.)   tensor(0.1619)


In [37]:
diff_h1 = (mean_h1_class1 - mean_h1_class0)
print(diff_h1)
score_h1 = diff_h1.abs()

values, indices = torch.sort(
    score_h1, descending=True
)

for rank in range(10):
    neuron_idx = indices[rank].item()
    score = values[rank].item()
    difference = diff_h1[neuron_idx].item()

    print(f"Rank {rank+1:02d} | Neuron {neuron_idx:03d} | score {score:0.4f} Difference {difference:+.4f}")

tensor([ 1.3818e-01,  1.8277e-01,  0.0000e+00, -3.0735e-01,  0.0000e+00,
        -1.9150e-01,  0.0000e+00, -4.2949e-03, -1.2129e-01,  1.7650e-01,
        -1.7287e-04, -2.9561e-04, -2.3929e-05, -7.4460e-02, -4.1905e-05,
         4.6091e-01, -1.7099e-01, -1.8362e-01,  0.0000e+00, -4.0783e-03,
        -2.5517e-01, -1.6650e-04,  0.0000e+00,  0.0000e+00, -8.9676e-02,
         0.0000e+00,  1.3391e-02,  0.0000e+00,  0.0000e+00, -1.6534e-01,
         0.0000e+00,  1.4307e-01,  0.0000e+00,  4.2858e-01, -2.0842e-01,
         3.8955e-01,  3.1113e-01, -3.4326e-02,  1.6623e-01,  0.0000e+00,
         0.0000e+00, -9.6296e-04,  0.0000e+00,  0.0000e+00,  0.0000e+00,
        -1.6793e-01,  3.3922e-01,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00, -2.6564e-04,  1.3287e-01, -1.6103e-02,  0.0000e+00,
        -7.4126e-06, -2.3150e-01,  3.4751e-01,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00, -1.3649e-01,  0.0000e+00, -2.9966e-01,
         0.0000e+00, -4.0636e-03,  0.0000e+00,  0.0

In [41]:
print(h1.shape)
num_neurons = h1.shape[1]
top_k = max(1, int(num_neurons * 0.03))
print("Number of neurons: ", num_neurons)
print("Top 3%: ", top_k)

top_values, top_indices = torch.topk(
    score_h1,
    k = top_k
)
print("Top 3% neurons: ", top_indices)
print("Scores: ", top_values)

torch.Size([1000, 100])
Number of neurons:  100
Top 3%:  3
Top 3% neurons:  tensor([15, 33, 35])
Scores:  tensor([0.4609, 0.4286, 0.3895])
